In [ ]:
!pip install --upgrade pytorch-forecasting pytorch-lightning


In [ ]:
import pytorch_lightning
import pytorch_forecasting
print(pytorch_lightning.__version__)
print(pytorch_forecasting.__version__)

2.5.3
1.4.0


In [ ]:
import numpy as np
from sklearn.preprocessing import RobustScaler

"""
Прави трансофрмации на лагираните вредности (не користи директни класични лагови за да се избегне висока корелација со таргетот).
Се прават различни математички операции како Логаритамска промена, разлика во цена за да се добијат дополнителни параметри
кои ќе го подобрат моделот, а притоа да не се со висока кореалција со таргетот како обичните лагирани вредности.
Дополнително сите вакви лагови се склаирани со RobustScaler (најдобри резултатит даде со тој, пробано е и со log, minmax)

 """


def add_lagged_features(df, target_col='close', max_lag=10):
    df = df.copy()
    for lag in range(1, max_lag+1):
        df[f'return_{lag}'] = (df[target_col] / df[target_col].shift(lag)) - 1
        df[f'log_return_{lag}'] = np.log(df[target_col] / df[target_col].shift(lag))
        df[f'momentum_{lag}'] = df[target_col] - df[target_col].shift(lag)
    # Optionally scale returns and momentum
    for base in ['return', 'log_return', 'momentum']:
        for lag in range(1, max_lag+1):
            col = f'{base}_{lag}'
            if col in df.columns:
                scaler = RobustScaler()
                df[f'{col}_robust'] = scaler.fit_transform(df[[col]].fillna(0))
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(method='ffill', limit=3).dropna()
    return df


In [ ]:
# Import required libraries

# !pip install pytorch_forecasting
import os
import json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# PyTorch Lightning imports with compatibility handling
try:
    from pytorch_lightning import Trainer, LightningModule
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
    from pytorch_lightning.loggers import TensorBoardLogger
except ImportError:
    from lightning.pytorch import Trainer, LightningModule
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
    from lightning.pytorch.loggers import TensorBoardLogger

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, SMAPE

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Check PyTorch version
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

CUDA available: True
GPU: Tesla T4
GPU Memory: 14.7 GB
PyTorch version: 2.8.0+cu126
CUDA version: 12.6


In [ ]:
# 🎯 SMART FEATURE CURATION - Fix Overfitting & Data Leakage
def apply_smart_feature_curation(df, target_col='close', correlation_threshold=0.99):
    """
   Го чисти датасетот, односно ги остранува сите параметри кои имат висока кореалација со таргетот или меѓу себе,
    корелација над 0.99 (пробано е лимитот да биде 0.95/0.90, најдобри рез на 0.99)

    """

    print(f"🎯 === SMART FEATURE CURATION ===")
    print(f"Target: {target_col}")
    print(f"Original features: {df.shape[1]}")
    print(f"Correlation threshold: {correlation_threshold}")

    df_clean = df.copy()

    if target_col not in df_clean.columns:
        raise ValueError(f"Target column '{target_col}' not found in data")

    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

    essential_cols = [target_col, 'time_idx', 'group']
    feature_cols = [col for col in numeric_cols if col not in essential_cols]

    print(f"Analyzing {len(feature_cols)} feature columns for curation...")

    print(f"\n🔍 STEP 1: Target Leakage Detection")
    target_correlations = df_clean[feature_cols + [target_col]].corr()[target_col].abs()

    leaked_features = []
    for feature in feature_cols:
        if feature in target_correlations.index:
            correlation = target_correlations[feature]
            if correlation >= correlation_threshold:
                print(f"  ❌ LEAKED: {feature} has {correlation:.6f} correlation with target!")
                leaked_features.append(feature)
            elif correlation >= 0.95:
                print(f"  ⚠️  HIGH: {feature} has {correlation:.6f} correlation with target")
            elif correlation >= 0.8:
                print(f"  📊 MODERATE: {feature} has {correlation:.6f} correlation with target")

    if leaked_features:
        print(f"\n🚫 Removing {len(leaked_features)} leaked features: {leaked_features}")
        feature_cols = [col for col in feature_cols if col not in leaked_features]
        df_clean = df_clean.drop(columns=leaked_features)
    else:
        print(f"✅ No target leakage detected!")

    print(f"\n🔍 STEP 2: Feature-to-Feature Correlation Analysis")

    if len(feature_cols) > 1:
        feature_corr_matrix = df_clean[feature_cols].corr().abs()

        # Find highly correlated pairs
        highly_correlated_pairs = []
        features_to_remove = set()

        for i in range(len(feature_cols)):
            for j in range(i + 1, len(feature_cols)):
                feature1 = feature_cols[i]
                feature2 = feature_cols[j]
                correlation = feature_corr_matrix.loc[feature1, feature2]

                if correlation >= correlation_threshold:
                    highly_correlated_pairs.append((feature1, feature2, correlation))
                    print(f"  ❌ REDUNDANT: {feature1} <-> {feature2}: {correlation:.6f}")

                    # Keep the feature with higher target correlation
                    if feature1 in target_correlations.index and feature2 in target_correlations.index:
                        target_corr1 = target_correlations[feature1]
                        target_corr2 = target_correlations[feature2]

                        if target_corr1 >= target_corr2:
                            features_to_remove.add(feature2)
                            print(f"    → Removing {feature2} (target corr: {target_corr2:.4f})")
                            print(f"    → Keeping {feature1} (target corr: {target_corr1:.4f})")
                        else:
                            features_to_remove.add(feature1)
                            print(f"    → Removing {feature1} (target corr: {target_corr1:.4f})")
                            print(f"    → Keeping {feature2} (target corr: {target_corr2:.4f})")
                    else:
                        # If target correlation not available, remove the second one
                        features_to_remove.add(feature2)
                        print(f"    → Removing {feature2} (default choice)")

        # Remove redundant features
        if features_to_remove:
            print(f"\n🚫 Removing {len(features_to_remove)} redundant features:")
            for feature in features_to_remove:
                print(f"  - {feature}")
            df_clean = df_clean.drop(columns=list(features_to_remove))
            feature_cols = [col for col in feature_cols if col not in features_to_remove]
        else:
            print(f"✅ No redundant features found!")

    # Step 3: Final validation
    print(f"\n✅ STEP 3: Final Validation")

    # Check remaining target correlations
    if len(feature_cols) > 0:
        final_target_corr = df_clean[feature_cols + [target_col]].corr()[target_col].abs()
        max_target_corr = final_target_corr[feature_cols].max()
        print(f"  Maximum remaining target correlation: {max_target_corr:.6f}")

        if max_target_corr >= correlation_threshold:
            print(f"  ⚠️  WARNING: Still have high target correlation!")
        else:
            print(f"  ✅ SUCCESS: No high correlations remain!")

        # Check remaining feature-to-feature correlations
        if len(feature_cols) > 1:
            final_feature_corr = df_clean[feature_cols].corr().abs()
            # Get upper triangle (excluding diagonal)
            upper_triangle = final_feature_corr.where(
                np.triu(np.ones(final_feature_corr.shape), k=1).astype(bool)
            )
            max_feature_corr = upper_triangle.max().max()
            print(f"  Maximum remaining feature-feature correlation: {max_feature_corr:.6f}")

            if max_feature_corr >= correlation_threshold:
                print(f"  ⚠️  WARNING: Still have redundant features!")
            else:
                print(f"  ✅ SUCCESS: No redundant features remain!")

    # Summary
    print(f"\n🎉 === CURATION COMPLETE ===")
    print(f"  Original features: {df.shape[1]}")
    print(f"  Final features: {df_clean.shape[1]}")
    print(f"  Features removed: {df.shape[1] - df_clean.shape[1]}")
    print(f"  Remaining feature columns: {len(feature_cols)}")

    if len(feature_cols) > 0:
        print(f"  Feature diversity: ✅ GOOD")
        print(f"  Expected R² improvement: 🚀 MAJOR (from -3.2040 to positive)")
    else:
        print(f"  ⚠️  WARNING: No features remaining after curation!")

    return df_clean

In [ ]:
# Configuration
def get_tft_config():
    """конфигурацијата на моделот во зависност од графичката на која се тренира, финалната користена верзија е L4 """

    # Auto-detect GPU for optimal settings
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"🚀 Optimizing for GPU: {gpu_name}")

    # Base configuration optimized for better GPUs
    if "A100" in gpu_name or "V100" in gpu_name:
        # High-end GPU settings
        config = {
            'max_prediction_length': 24,
            'max_encoder_length': 168,
            'batch_size': 128,
            'learning_rate': 0.001,
            'hidden_size': 96,
            'attention_head_size': 6,
            'dropout': 0.1,
            'hidden_continuous_size': 24,
            'max_epochs': 2,
            'gradient_clip_val': 1.0,
            'patience': 15,
            'min_delta': 0.0005,
            'precision': '16-mixed',
            'accumulate_grad_batches': 1,
        }
        print("🔥 Using A100/V100 optimized settings!")
    elif "L4" in gpu_name:

        config = {

    'max_prediction_length': 24,
    'max_encoder_length': 240,
    'batch_size': 120,
    'learning_rate': 0.0003,
    'hidden_size': 128,
    'attention_head_size': 8,
    'dropout': 0.1,
    'hidden_continuous_size': 22,
    'max_epochs': 75,
    'gradient_clip_val': 0.5,
    'patience': 12,
    'min_delta': 0.0001,
    'precision': '32',
    'accumulate_grad_batches': 1,
    'optimizer': 'adamw',
    'weight_decay': 1e-4,


    'lr_scheduler': 'reduce_on_plateau',
    'lr_scheduler_patience': 8,
    'lr_scheduler_factor': 0.7,
    'lr_scheduler_min_lr': 1e-6,
}
        print("🚀 Using L4 optimized settings - excellent efficiency!")
    elif "T4" in gpu_name:

        config = {
   "max_prediction_length": 48,
  "max_encoder_length": 216,
  "batch_size": 120,
  "learning_rate": 0.0003,
  "hidden_size": 96,
  "attention_head_size": 8,
  "dropout": 0.15,
  "hidden_continuous_size": 22,
  "max_epochs": 50,
  "gradient_clip_val": 0.5,
  "patience": 12,
  "min_delta": 0.0001,
  "precision": "32",
  "accumulate_grad_batches": 1,
  "optimizer": "adamw",
  "weight_decay": 0.0001

}
        print("⚡ Using T4 optimized settings!")
    else:
        # Conservative settings for other/unknown GPUs
        config = {
            'max_prediction_length': 24,
            'max_encoder_length': 192,
            'batch_size': 120,
            'learning_rate': 0.001,
            'hidden_size': 88,
            'attention_head_size': 6,
            'dropout': 0.1,
            'hidden_continuous_size': 22,
            'max_epochs': 1,
            'gradient_clip_val': 1.0,
            'patience': 10,
            'min_delta': 0.001,
            'precision': '32',
            'accumulate_grad_batches': 1,
        }
        print("🛡️ Using conservative settings for unknown GPU")

    return config

config = get_tft_config()
print("Production-ready configuration loaded:")
for key, value in config.items():
    print(f"  {key}: {value}")

🚀 Optimizing for GPU: Tesla T4
⚡ Using T4 optimized settings!
Production-ready configuration loaded:
  max_prediction_length: 48
  max_encoder_length: 216
  batch_size: 120
  learning_rate: 0.0003
  hidden_size: 96
  attention_head_size: 8
  dropout: 0.15
  hidden_continuous_size: 22
  max_epochs: 50
  gradient_clip_val: 0.5
  patience: 12
  min_delta: 0.0001
  precision: 32
  accumulate_grad_batches: 1
  optimizer: adamw
  weight_decay: 0.0001


In [ ]:

def fix_data_leakage(df, target_column='close'):

    """ ги отстранува само оние карактеристики (features) што ја откриваат идната или тековната цена,
     односно податоци кои моделот не би ги имал на располагање во реални услови на предвидување.
     Ги издвојува таргето (close) ги остранува сите параметри oд датасетот преку кои моделот може на памет да уче (overfit)
      како future_, next_, tomorrow и ги отстранува обичните лагови (остануваат само предходно пресметаните лагирани вредности
"""
    def is_safe_feature(feature_name):
        feature_lower = feature_name.lower()

        # Always keep target column
        if feature_name == target_column:
            return True

        # Keep essential columns for TFT
        if feature_name in ['close', 'target', 'time_idx', 'group']:
            return True

        # Remove obvious future-looking features (more conservative)
        dangerous_keywords = [
            'future_', 'next_', 'tomorrow_', 'ahead_', 'forecast_',
            'predict_', 'target_shift', 'lead_', 'forward_'
        ]

        # Remove features that are clearly future-looking
        if any(keyword in feature_lower for keyword in dangerous_keywords):
            return False

        # Remove same-period features of the target if they're not lagged
        if feature_lower in [target_column.lower()] and feature_name != target_column:
            return False

        # Remove features that are just shifted versions of target without proper lag
        if 'shift' in feature_lower and target_column.lower() in feature_lower:
            # Only allow if it's clearly a lag (negative shift)
            import re
            shift_match = re.search(r'shift_?(-?\d+)', feature_lower)
            if shift_match:
                shift_value = int(shift_match.group(1))
                if shift_value >= 0:  # Future or current period
                    return False

        # Keep most other features - they're likely safe
        return True

    original_cols = len(df.columns)
    safe_features = [f for f in df.columns if is_safe_feature(f)]
    df_clean = df[safe_features].copy()

    print(f"Data leakage prevention: {original_cols} -> {len(safe_features)} features")
    removed_features = [f for f in df.columns if f not in safe_features]
    if removed_features:
        print(f"Removed features: {removed_features[:10]}{'...' if len(removed_features) > 10 else ''}")

    return df_clean

In [ ]:
# # 🚀 MOMENTUM INDICATORS: Feature Engineering for Positive R²
# def add_momentum_indicators(df, target_col='close'):
#     """Add powerful momentum indicators to push R² from -0.39 to positive"""

#     print("🚀 Adding momentum indicators...")

#     df = df.copy()

#     # Ensure we have the required price data
#     if target_col not in df.columns:
#         raise ValueError(f"Target column '{target_col}' not found")

#     # 1. PRICE MOMENTUM INDICATORS
#     # Rate of Change (ROC) - multiple timeframes
#     df['roc_5'] = df[target_col].pct_change(5) * 100
#     df['roc_10'] = df[target_col].pct_change(10) * 100
#     df['roc_20'] = df[target_col].pct_change(20) * 100

#     # Momentum (price difference)
#     df['momentum_5'] = df[target_col] - df[target_col].shift(5)
#     df['momentum_10'] = df[target_col] - df[target_col].shift(10)
#     df['momentum_20'] = df[target_col] - df[target_col].shift(20)

#     # 2. ACCELERATION INDICATORS
#     # Price acceleration (second derivative)
#     df['price_accel_5'] = df['roc_5'] - df['roc_5'].shift(5)
#     df['price_accel_10'] = df['roc_10'] - df['roc_10'].shift(10)

#     # Velocity momentum (change in momentum)
#     df['momentum_velocity_5'] = df['momentum_5'] - df['momentum_5'].shift(5)
#     df['momentum_velocity_10'] = df['momentum_10'] - df['momentum_10'].shift(10)

#     # 3. TREND STRENGTH INDICATORS
#     # Moving average convergence/divergence signals
#     if 'ad_line' in df.columns:
#         df['ad_momentum_5'] = df['ad_line'] - df['ad_line'].shift(5)
#         df['ad_momentum_10'] = df['ad_line'] - df['ad_line'].shift(10)

#     if 'vpt_ma_10' in df.columns:
#         df['vpt_momentum_5'] = df['vpt_ma_10'] - df['vpt_ma_10'].shift(5)
#         df['vpt_momentum_10'] = df['vpt_ma_10'] - df['vpt_ma_10'].shift(10)

#     # 4. VOLATILITY-ADJUSTED MOMENTUM
#     # Rolling standard deviation for normalization
#     rolling_std_10 = df[target_col].rolling(10).std()
#     rolling_std_20 = df[target_col].rolling(20).std()

#     # Volatility-adjusted momentum (momentum/volatility)
#     df['vol_adj_momentum_10'] = df['momentum_10'] / (rolling_std_10 + 1e-8)
#     df['vol_adj_momentum_20'] = df['momentum_20'] / (rolling_std_20 + 1e-8)

#     # Relative momentum strength
#     df['rel_momentum_strength'] = df['momentum_5'] / (df['momentum_20'] + 1e-8)

#     # 5. MOMENTUM PERSISTENCE INDICATORS
#     # Momentum direction consistency
#     df['momentum_direction_5'] = (df['momentum_5'] > 0).astype(int)
#     df['momentum_direction_10'] = (df['momentum_10'] > 0).astype(int)

#     # Momentum consistency score (how often momentum is in same direction)
#     df['momentum_consistency_5'] = df['momentum_direction_5'].rolling(5).mean()
#     df['momentum_consistency_10'] = df['momentum_direction_10'].rolling(10).mean()

#     # 6. CROSS-TIMEFRAME MOMENTUM
#     # Short vs long momentum ratio
#     df['momentum_ratio_5_20'] = df['momentum_5'] / (df['momentum_20'] + 1e-8)
#     df['momentum_ratio_10_20'] = df['momentum_10'] / (df['momentum_20'] + 1e-8)

#     # Momentum divergence (when short and long disagree)
#     df['momentum_divergence'] = np.abs(df['momentum_5'] - df['momentum_20']) / (np.abs(df['momentum_20']) + 1e-8)

#     # 7. MOMENTUM EXTREME DETECTION
#     # Momentum z-scores (standardized momentum)
#     df['momentum_zscore_5'] = (df['momentum_5'] - df['momentum_5'].rolling(50).mean()) / (df['momentum_5'].rolling(50).std() + 1e-8)
#     df['momentum_zscore_10'] = (df['momentum_10'] - df['momentum_10'].rolling(50).mean()) / (df['momentum_10'].rolling(50).std() + 1e-8)

#     # Momentum extreme flags
#     df['momentum_extreme_pos'] = (df['momentum_zscore_10'] > 2).astype(int)
#     df['momentum_extreme_neg'] = (df['momentum_zscore_10'] < -2).astype(int)

#     # Remove any infinite values
#     df = df.replace([np.inf, -np.inf], np.nan)

#     # Forward fill small gaps, then drop remaining NaN
#     df = df.fillna(method='ffill', limit=3).dropna()

#     return df


# print("✅ Momentum indicators now integrated into main loading function!")
# print("🚀 Use load_and_prepare_data_fixed() with add_momentum_indicators=True!")

In [ ]:
def load_and_prepare_data_fixed(data_path, metadata_path=None, use_curated_features=True, use_momentum_indicators=False):
    """го спрема датасетот, го определува таргетот, ја повикува apply_smart_feature_curation да го изчисти од параметри
     кои имаат висока корелација меѓунив/со таргето. тука се додаваат и momentum индикатори (во финалната верзија се исклучени,
     подобри резултати даваше без нив).
"""

    import pandas as pd
    import json
    import os

    print("💯 === SIMPLIFIED DATA LOADING WITH CURATED FEATURES ONLY ===")
    # Load data
    df = pd.read_csv(data_path)
    print(f"Loaded data shape: {df.shape}")

    # BULLETPROOF: Force target to 'close' - no exceptions!
    target_col = 'close'

    # Create clean metadata
    metadata = {
        'target': 'close',
        'target_column': 'close',
        'original_target_column': 'close'
    }

    print(f"💯 BULLETPROOF: Target FORCED to: {target_col}")

    # Verify target column exists
    if target_col not in df.columns:
        available_cols = list(df.columns)
        raise ValueError(f"Target column '{target_col}' not found in data. Available columns: {available_cols[:10]}")

    # Add required columns if missing
    if 'time_idx' not in df.columns:
        df['time_idx'] = range(len(df))
        print("Added time_idx column")

    if 'group' not in df.columns:
        df['group'] = 0
        print("Added group column")

    # 🎯 CURATED FEATURE SELECTION - Fix overfitting issue
    if use_curated_features:


        # Apply correlation-based feature curation
        df_curated = apply_smart_feature_curation(df, target_col)
        print(f"📊 After curation: {df_curated.shape} (from {df.shape})")

        df = df_curated
        metadata['curated_features'] = True
        metadata['curation_applied'] = True
    else:
        metadata['curated_features'] = False

    # 🚀 MOMENTUM INDICATORS - DISABLED BY DEFAULT
    if use_momentum_indicators:
        print("\n🚀 === ADDING MOMENTUM INDICATORS ===")
        print("⚠️  WARNING: This will add 25+ features and may cause issues!")

        # You would need to implement add_momentum_indicators function
        # For now, just skip this to get back to working baseline
        print("❌ SKIPPED: Momentum indicators disabled to prevent issues")
        metadata['momentum_indicators'] = False
        metadata['momentum_applied'] = False
    else:
        print("\n✅ MOMENTUM INDICATORS: Disabled (using curated features only)")
        metadata['momentum_indicators'] = False

    print(f"Final data shape: {df.shape}")
    print(f"💯 BULLETPROOF: Final target confirmed as: {target_col}")

    # Summary of what was applied
    applied_features = []
    if use_curated_features:
        applied_features.append("✅ Curated features (removed data leakage)")

    if applied_features:
        print(f"\n🎉 ENHANCEMENTS APPLIED:")
        for feature in applied_features:
            print(f"  {feature}")
        print("🚀 Expected: Much better than R² = -3.2040!")
    else:
        print("🎯 Basic data loading - no enhancements applied")

    return df, metadata

In [ ]:
#  OVERFITTING PREVENTION: Remove highly correlated features
# def remove_correlated_features(df, target_col, correlation_threshold=0.95):
#     """Remove features that are too highly correlated to prevent overfitting"""

#     print(f"🛡️ === OVERFITTING PREVENTION ===")
#     print(f"Removing features with correlation > {correlation_threshold}")

#     # Get feature columns (exclude target, time_idx, group)
#     feature_cols = [col for col in df.columns if col not in [target_col, 'time_idx', 'group']]

#     # Calculate correlation matrix
#     feature_df = df[feature_cols].select_dtypes(include=[np.number])
#     corr_matrix = feature_df.corr().abs()

#     # Find highly correlated pairs
#     high_corr_pairs = []
#     removed_features = set()

#     for i in range(len(corr_matrix.columns)):
#         for j in range(i+1, len(corr_matrix.columns)):
#             col1 = corr_matrix.columns[i]
#             col2 = corr_matrix.columns[j]
#             correlation = corr_matrix.iloc[i, j]

#             if correlation > correlation_threshold:
#                 high_corr_pairs.append((col1, col2, correlation))

#                 # Keep the one with higher correlation to target
#                 target_corr1 = abs(df[col1].corr(df[target_col]))
#                 target_corr2 = abs(df[col2].corr(df[target_col]))

#                 if target_corr1 >= target_corr2:
#                     removed_features.add(col2)
#                     print(f"  ❌ Removing {col2} (corr with {col1}: {correlation:.3f})")
#                 else:
#                     removed_features.add(col1)
#                     print(f"  ❌ Removing {col1} (corr with {col2}: {correlation:.3f})")

#     # Keep features that weren't removed
#     kept_features = [col for col in feature_cols if col not in removed_features]

#     # Add back essential columns
#     final_columns = [target_col, 'time_idx', 'group'] + kept_features
#     df_filtered = df[final_columns].copy()

#     print(f"\n📊 Correlation filtering results:")
#     print(f"  Original features: {len(feature_cols)}")
#     print(f"  Highly correlated pairs found: {len(high_corr_pairs)}")
#     print(f"  Features removed: {len(removed_features)}")
#     print(f"  Features kept: {len(kept_features)}")
#     print(f"  Final dataset shape: {df_filtered.shape}")

#     return df_filtered, kept_features

# print("✅ Correlation filter function ready!")
# print("This will prevent overfitting by removing redundant features")

✅ Correlation filter function ready!
This will prevent overfitting by removing redundant features


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
def select_top_features(df, target_col, n_features=60, corr_threshold=0.95):
    """
   избира n најважни параметри (features) од датасетот сет врз основа на корелација со таргетот, при тоа избегнувајќи редундантни карактеристики.
   Лаговите дополнително се додаваат и затоа има потреба од дополнителна функција за проверка на корелација со таргетот.

    """
    numeric_df = df.select_dtypes(include=[np.number])
    correlations = numeric_df.corr()[target_col].abs()
    correlations = correlations.dropna().sort_values(ascending=False)
    essential_cols = [target_col, 'group']
    available_features = [col for col in correlations.index if col not in essential_cols]
    print(f"IMPORTANT: Target column '{target_col}' excluded from features")
    print(f"Including time_idx as a feature.")
    # 1. Top correlation features (50% of quota, but filter redundancy)
    correlation_features = []
    for f in available_features:
        if all(abs(numeric_df[f].corr(numeric_df[cf])) < corr_threshold for cf in correlation_features):
            correlation_features.append(f)
        if len(correlation_features) >= int(n_features * 0.5):
            break
    print(f"Selected {len(correlation_features)} top-correlation features")
    # 2. High variance features from remaining (50% of quota, filter redundancy)
    remaining_features = [f for f in available_features if f not in correlation_features]
    feature_variances = numeric_df[remaining_features].var().sort_values(ascending=False) if remaining_features else []
    variance_features = []
    for f in feature_variances.index if hasattr(feature_variances, 'index') else []:
        if all(abs(numeric_df[f].corr(numeric_df[cf])) < corr_threshold for cf in correlation_features + variance_features):
            variance_features.append(f)
        if len(variance_features) >= int(n_features * 0.5):
            break
    print(f"Selected {len(variance_features)} high-variance features")
    top_features = correlation_features + variance_features
    top_features = top_features[:n_features]
    selected_features = essential_cols + top_features
    print(f"\nOPTIMIZED feature selection complete:")
    print(f"  Essential features: {len(essential_cols)}")
    print(f"  Top-correlation: {len(correlation_features)}")
    print(f"  High-variance: {len(variance_features)}")
    print(f"  Total selected: {len(selected_features)}")
    # Show which lagged features are included
    lagged = [f for f in top_features if any(x in f for x in ['return_', 'log_return_', 'momentum_'])]
    print(f"Lagged/return/log_return/momentum features included: {lagged}")
    return df[selected_features].copy(), top_features

In [ ]:
import numpy as np
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.data.encoders import NaNLabelEncoder

def create_tft_datasets(df, config, target_col, metadata):

    """Подготовка на датасет за TFT модел. Отрансува колони со голем број nan/missing values ,прави класификација на паремтрите, се делат на категориски и континурани.
    (категориските се енкодирани со NaNLabelEncoder). Се разделуваат статичките временски променливи (што не се менуваат во текот на времето) и тие што немаат констани вредности
     (карактеристично за TFT моделите.) Се дели датасетот на train/valid, врз основа на бројот на секвенци и вкупниот број на податоци,
      нема фиксен број како 80/20 итн. min за valid е 20%, маx 40% oд датасетот. Дополнително се користи TimeSeriesDataSet за структурирање на податоците за временски серии.
       Со GroupNormalizer се нормализира таргетот и неколку континуирани параметри, пресметаните лагови се нормализирани со RobustScaler.
"""

    print(f"Creating TFT datasets with target column: {target_col}")
    print(f"Data shape: {df.shape}")

    # Ensure required columns exist
    if "time_idx" not in df.columns:
        df["time_idx"] = range(len(df))
        print("Created time_idx column")

    if "group" not in df.columns:
        df["group"] = 0  # Single time series
        print("Created group column for single series")

    # Ensure target column exists
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in data")

    # Clean data - remove any NaN or infinite values
    original_len = len(df)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    if len(df) < original_len:
        print(f"Removed {original_len - len(df)} rows with NaN/infinite values")

    # Ensure we have enough data
    min_required = config["max_encoder_length"] + config["max_prediction_length"]
    if len(df) < min_required:
        raise ValueError(
            f"Not enough data. Need at least {min_required} rows, got {len(df)}"
        )

    # Explicit feature classification
    categorical_whitelist = ["ticker", "sector", "industry", "exchange", "country","gdp_release"]
    continuous_whitelist = [
        "news_frequency_daily",
        "month_cos",
        "month_sin",
        "days_since_fed_meeting",
        "days_to_fed_meeting",
        "dom_sin",
        "dow_sin",
        "dow_cos",
        "hour_cos",
    ]

    categorical_features = []
    continuous_features = []

    print("Assigning features into categorical and continuous...")

    for col in df.columns:
        if col in ["time_idx", "group", target_col]:
            continue
        elif col in categorical_whitelist:
            df[col] = df[col].astype(str).fillna("__missing__")
            categorical_features.append(col)
        elif col in continuous_whitelist:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            continuous_features.append(col)
        else:
            # default logic: if numeric → continuous, else categorical
            if pd.api.types.is_numeric_dtype(df[col]):
                continuous_features.append(col)
            else:
                df[col] = df[col].astype(str).fillna("__missing__")
                categorical_features.append(col)

    print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
    print(f"Continuous features ({len(continuous_features)}): {continuous_features}")

    # Enhanced static features detection
    static_categoricals = []
    static_reals = []

    # Detect time-invariant features (static features)
    for col in categorical_features:
        if df.groupby("group")[col].nunique().max() == 1:
            static_categoricals.append(col)

    for col in continuous_features:
        if df.groupby("group")[col].nunique().max() == 1:
            static_reals.append(col)

    # Remove static features from time-varying lists
    time_varying_known_categoricals = [
        col for col in categorical_features if col not in static_categoricals
    ]
    time_varying_known_reals = [
        col for col in continuous_features if col not in static_reals
    ]

    print(f"Static categoricals: {static_categoricals}")
    print(f"Static reals: {static_reals}")
    print(f"Time-varying categoricals: {time_varying_known_categoricals}")
    print(f"Time-varying reals: {time_varying_known_reals}")

    # Train/validation split
    total_samples = len(df)
    min_encoder_decoder_length = (
        config["max_encoder_length"] + config["max_prediction_length"]
    )
    min_validation_sequences = 500
    samples_per_sequence = min_encoder_decoder_length
    min_val_samples = min_validation_sequences * samples_per_sequence
    min_val_samples = max(min_val_samples, int(total_samples * 0.20))
    max_val_samples = int(total_samples * 0.40)
    val_samples = min(min_val_samples, max_val_samples)
    training_cutoff = total_samples - val_samples - samples_per_sequence

    print(f"Training cutoff at index: {training_cutoff}")
    print(f"Training samples: {training_cutoff}")
    print(f"Validation samples: {val_samples}")
    print(
        f"Validation percentage: {(val_samples/total_samples)*100:.1f}%"
    )

    # Create categorical encoders
    categorical_encoders = {
        col: NaNLabelEncoder(add_nan=True)
        for col in time_varying_known_categoricals + static_categoricals
    }

    # Training dataset
    training_dataset = TimeSeriesDataSet(
        df[lambda x: x.time_idx <= training_cutoff],
        time_idx="time_idx",
        target=target_col,
        group_ids=["group"],
        max_encoder_length=config["max_encoder_length"],
        max_prediction_length=config["max_prediction_length"],
        static_categoricals=static_categoricals,
        static_reals=static_reals,
        time_varying_known_categoricals=time_varying_known_categoricals,
        time_varying_known_reals=time_varying_known_reals,
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=[target_col],
        categorical_encoders=categorical_encoders,
        target_normalizer=GroupNormalizer(groups=["group"], transformation=None),
        scalers={
            col: GroupNormalizer(groups=["group"], transformation=None)
            for col in time_varying_known_reals[:15]
        },
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True,
        min_encoder_length=config["max_encoder_length"] // 4,
        min_prediction_length=1,
    )

    print("✓ Training dataset created successfully")

    # Validation dataset
    validation_start = training_cutoff + 1
    validation_data = df[df.time_idx >= validation_start].copy()

    validation_dataset = TimeSeriesDataSet(
        validation_data,
        time_idx="time_idx",
        target=target_col,
        group_ids=["group"],
        max_encoder_length=config["max_encoder_length"],
        max_prediction_length=config["max_prediction_length"],
        static_categoricals=static_categoricals,
        static_reals=static_reals,
        time_varying_known_categoricals=time_varying_known_categoricals,
        time_varying_known_reals=time_varying_known_reals,
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=[target_col],
        categorical_encoders=categorical_encoders,
        target_normalizer=training_dataset.target_normalizer,
        scalers=training_dataset.scalers,
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True,
        min_encoder_length=config["max_encoder_length"] // 4,
        min_prediction_length=1,
    )

    print("✓ Validation dataset created successfully")

    # Dataset statistics
    print("\nDataset Statistics:")
    print(f"  Training samples: {len(training_dataset)}")
    print(f"  Validation samples: {len(validation_dataset)}")
    print(f"  Training cutoff used: {training_cutoff}")
    print(f"  Total features: {len(training_dataset.reals + training_dataset.categoricals)}")
    print(f"  Encoder length: {config['max_encoder_length']}")
    print(f"  Prediction length: {config['max_prediction_length']}")

    return training_dataset, validation_dataset


In [ ]:
def create_tft_model(training_dataset, config):
    """го зима конфигот и го креира моделот, се користи wrapper поради проблеми со компактабилност на lightning верзиите"""

    print("Creating production-ready TFT model...")

    # Use the new compatibility function
    model = create_compatible_tft_model(training_dataset, config)

    # Verify model structure
    print(f"\nModel Architecture Summary:")
    print(f"  - Hidden size: {config['hidden_size']}")
    print(f"  - Attention heads: {config['attention_head_size']}")
    print(f"  - Dropout: {config['dropout']}")
    print(f"  - Continuous hidden size: {config['hidden_continuous_size']}")
    print(f"  - Output quantiles: 7")
    print(f"  - Static categoricals: {len(training_dataset.static_categoricals)}")
    print(f"  - Static reals: {len(training_dataset.static_reals)}")
    print(f"  - Lightning compatible: {isinstance(model, LightningModule)}")

    return model

In [ ]:
# Lightning Compatibility Wrapper - DEVICE-AWARE VERSION
class TFTLightningWrapper(LightningModule):
    """
    Wrapper за TFT modelot.
    """

    def __init__(self, tft_model):
        super().__init__()
        # Use standard PyTorch module registration - this is much more reliable
        self.tft_model = tft_model

        # Don't try to save hyperparameters from the wrapped model - too risky
        # Lightning will handle hyperparameters for the wrapper itself
        print("✓ TFT model wrapped successfully")

    def forward(self, x):
        """Forward pass - delegates to wrapped model with proper device handling"""
        # Ensure wrapped model is on same device as wrapper
        self._sync_devices()
        return self.tft_model(x)

    def training_step(self, batch, batch_idx):
        """Training step - delegates to wrapped model with device sync and logging context"""
        # Ensure wrapped model is on same device as wrapper
        self._sync_devices()

        # CRITICAL: Set up proper logging context for wrapped model
        original_current_fx_name = getattr(self.tft_model, '_current_fx_name', None)
        original_trainer = getattr(self.tft_model, '_trainer', None)

        try:
            # Set the logging context from our wrapper
            self.tft_model._current_fx_name = getattr(self, '_current_fx_name', 'training_step')
            self.tft_model._trainer = getattr(self, '_trainer', self.trainer)

            # Call the training step with proper context
            result = self.tft_model.training_step(batch, batch_idx)

            # Handle the logging manually if needed
            if isinstance(result, dict):
                # Log the metrics through our wrapper's context
                for key, value in result.items():
                    if 'loss' in key.lower() or 'metric' in key.lower():
                        try:
                            self.log(key, value, on_step=True, on_epoch=True, prog_bar=True)
                        except:
                            pass  # Skip if logging fails

            return result

        finally:
            # Restore original context
            if original_current_fx_name is not None:
                self.tft_model._current_fx_name = original_current_fx_name
            else:
                if hasattr(self.tft_model, '_current_fx_name'):
                    delattr(self.tft_model, '_current_fx_name')

            if original_trainer is not None:
                self.tft_model._trainer = original_trainer
            else:
                if hasattr(self.tft_model, '_trainer'):
                    self.tft_model._trainer = getattr(self, '_trainer', None)

    def validation_step(self, batch, batch_idx):
        """Validation step - delegates to wrapped model with device sync and logging context"""
        # Ensure wrapped model is on same device as wrapper
        self._sync_devices()

        # CRITICAL: Set up proper logging context for wrapped model
        # Temporarily assign our trainer's logging context to the wrapped model
        original_current_fx_name = getattr(self.tft_model, '_current_fx_name', None)
        original_trainer = getattr(self.tft_model, '_trainer', None)

        try:
            # Set the logging context from our wrapper
            self.tft_model._current_fx_name = getattr(self, '_current_fx_name', 'validation_step')
            self.tft_model._trainer = getattr(self, '_trainer', self.trainer)

            # Now call the validation step with proper context
            result = self.tft_model.validation_step(batch, batch_idx)

            # Handle the logging manually if needed
            if isinstance(result, dict):
                # Log the metrics through our wrapper's context
                for key, value in result.items():
                    if 'loss' in key.lower() or 'metric' in key.lower():
                        try:
                            self.log(key, value, on_step=False, on_epoch=True, prog_bar=True)
                        except:
                            pass  # Skip if logging fails

            return result

        finally:
            # Restore original context
            if original_current_fx_name is not None:
                self.tft_model._current_fx_name = original_current_fx_name
            else:
                if hasattr(self.tft_model, '_current_fx_name'):
                    delattr(self.tft_model, '_current_fx_name')

            if original_trainer is not None:
                self.tft_model._trainer = original_trainer
            else:
                if hasattr(self.tft_model, '_trainer'):
                    self.tft_model._trainer = getattr(self, '_trainer', None)

    def configure_optimizers(self):
        """Configure optimizers - delegates to wrapped model"""
        return self.tft_model.configure_optimizers()

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        """Prediction step - delegates to wrapped model if available"""
        # Ensure wrapped model is on same device as wrapper
        self._sync_devices()
        if hasattr(self.tft_model, 'predict_step'):
            return self.tft_model.predict_step(batch, batch_idx, dataloader_idx)
        else:
            # Fallback to forward pass
            return self(batch[0])

    def on_train_epoch_end(self):
        """End of training epoch - delegates to wrapped model if available"""
        if hasattr(self.tft_model, 'on_train_epoch_end'):
            return self.tft_model.on_train_epoch_end()

    def on_validation_epoch_end(self):
        """End of validation epoch - delegates to wrapped model if available"""
        if hasattr(self.tft_model, 'on_validation_epoch_end'):
            return self.tft_model.on_validation_epoch_end()

    def _sync_devices(self):
        """Ensure wrapped model is on the same device as the wrapper"""
        wrapper_device = next(self.parameters()).device
        tft_device = next(self.tft_model.parameters()).device

        if wrapper_device != tft_device:
            print(f"Device sync: Moving TFT model from {tft_device} to {wrapper_device}")
            self.tft_model = self.tft_model.to(wrapper_device)

    def _sync_trainer(self):
        """Ensure wrapped model has access to the trainer"""
        if hasattr(self, '_trainer') and self._trainer is not None:
            # Set trainer reference on wrapped model
            self.tft_model._trainer = self._trainer

    @property
    def trainer(self):
        """Override trainer property to ensure proper delegation"""
        if hasattr(self, '_trainer'):
            return self._trainer
        return None

    @trainer.setter
    def trainer(self, value):
        """Override trainer setter to sync with wrapped model"""
        self._trainer = value
        # Also set the trainer on the wrapped model
        if hasattr(self.tft_model, '_trainer') or hasattr(self.tft_model, 'trainer'):
            self.tft_model._trainer = value

    def on_fit_start(self):
        """Called when fit begins - sync trainer reference"""
        self._sync_trainer()
        if hasattr(self.tft_model, 'on_fit_start'):
            return self.tft_model.on_fit_start()

    def on_validation_start(self):
        """Called when validation begins - sync trainer reference"""
        self._sync_trainer()
        if hasattr(self.tft_model, 'on_validation_start'):
            return self.tft_model.on_validation_start()

    def to(self, device):
        """Override to method to ensure both wrapper and wrapped model move together"""
        super().to(device)
        self.tft_model = self.tft_model.to(device)
        return self

    def cuda(self, device=None):
        """Override cuda method to ensure both wrapper and wrapped model move together"""
        super().cuda(device)
        self.tft_model = self.tft_model.cuda(device)
        return self

    def cpu(self):
        """Override cpu method to ensure both wrapper and wrapped model move together"""
        super().cpu()
        self.tft_model = self.tft_model.cpu()
        return self

    def on_save_checkpoint(self, checkpoint):
        """
        Called when Lightning saves a checkpoint.
        Save the wrapped model's state and any necessary info to reconstruct it.
        """
        # Save the wrapped model's state
        checkpoint['tft_model_state_dict'] = self.tft_model.state_dict()

        # Save model architecture info if available
        if hasattr(self.tft_model, 'hparams'):
            checkpoint['tft_model_hparams'] = self.tft_model.hparams

        # Save model class info for reconstruction
        checkpoint['tft_model_class'] = self.tft_model.__class__.__name__

    def on_load_checkpoint(self, checkpoint):
        """
        Called when Lightning loads a checkpoint.
        Restore the wrapped model's state.
        """
        if 'tft_model_state_dict' in checkpoint:
            # Load the wrapped model's state
            self.tft_model.load_state_dict(checkpoint['tft_model_state_dict'])
            print("✅ Restored TFT model state from checkpoint")
        else:
            print("⚠️  No TFT model state found in checkpoint")

    @classmethod
    def load_from_checkpoint(cls, checkpoint_path, tft_model=None, **kwargs):
        """
        Custom checkpoint loading for TFTLightningWrapper.
        Requires the original tft_model to be provided.
        """
        if tft_model is None:
            raise ValueError(
                "TFTLightningWrapper.load_from_checkpoint requires 'tft_model' parameter. "
                "Please provide the original TemporalFusionTransformer model."
            )

        # Create new wrapper with the provided tft_model
        wrapper = cls(tft_model)

        # Load the checkpoint
        checkpoint = torch.load(checkpoint_path, map_location='cpu')

        # Load wrapper state
        if 'state_dict' in checkpoint:
            # Filter out tft_model parameters from state_dict to avoid conflicts
            wrapper_state_dict = {k: v for k, v in checkpoint['state_dict'].items()
                                if not k.startswith('tft_model.')}
            if wrapper_state_dict:
                wrapper.load_state_dict(wrapper_state_dict, strict=False)

        # Load TFT model state if available
        if 'tft_model_state_dict' in checkpoint:
            wrapper.tft_model.load_state_dict(checkpoint['tft_model_state_dict'])
            print("✅ Loaded TFT model state from checkpoint")

        return wrapper


def create_compatible_tft_model(training_dataset, config):
    """
    Create a TFT model with automatic Lightning compatibility wrapper

    Args:
        training_dataset: pytorch-forecasting TimeSeriesDataSet
        config: dictionary with model configuration

    Returns:
        Lightning-compatible model ready for Trainer.fit()
    """
    print("Creating TemporalFusionTransformer model...")

    # Create the original TFT model
    tft_model = TemporalFusionTransformer.from_dataset(
        training_dataset,
        learning_rate=config['learning_rate'],
        hidden_size=config['hidden_size'],
        attention_head_size=config['attention_head_size'],
        dropout=config['dropout'],
        hidden_continuous_size=config['hidden_continuous_size'],
        output_size=7,  # Quantile predictions
        loss=QuantileLoss(),
        log_interval=10,
        reduce_on_plateau_patience=4,
        static_categoricals=training_dataset.static_categoricals,
        static_reals=training_dataset.static_reals,
    )

    print(f"✓ Created TFT model with {sum(p.numel() for p in tft_model.parameters()):,} parameters")

    # Check if Lightning compatibility is needed
    is_lightning_compatible = isinstance(tft_model, LightningModule)
    print(f"Direct Lightning compatibility: {is_lightning_compatible}")

    if not is_lightning_compatible:
        print("🔧 Applying Lightning compatibility wrapper...")
        try:
            wrapped_model = TFTLightningWrapper(tft_model)
            print(f"✓ Wrapped model Lightning compatibility: {isinstance(wrapped_model, LightningModule)}")
            return wrapped_model
        except Exception as e:
            print(f"Primary wrapper failed: {e}")
            print("🔧 Trying minimal wrapper without hyperparameters...")

            # Fallback to minimal wrapper without hyperparameter handling
            class MinimalTFTWrapper(LightningModule):
                def __init__(self, model):
                    super().__init__()
                    self.tft_model = model  # Use same naming for consistency

                def forward(self, x):
                    return self.tft_model(x)

                def training_step(self, batch, batch_idx):
                    return self.tft_model.training_step(batch, batch_idx)

                def validation_step(self, batch, batch_idx):
                    return self.tft_model.validation_step(batch, batch_idx)

                def configure_optimizers(self):
                    return self.tft_model.configure_optimizers()

            minimal_wrapped = MinimalTFTWrapper(tft_model)
            print(f"✓ Minimal wrapper Lightning compatibility: {isinstance(minimal_wrapped, LightningModule)}")
            return minimal_wrapped
    else:
        print("✓ Model is already Lightning compatible")
        return tft_model


# Example usage function
def train_with_wrapper_example(model, train_dataloader, val_dataloader, config):
    """
    Example of how to use the wrapped model with PyTorch Lightning Trainer

    Args:
        model: TFT model (will be wrapped if needed)
        train_dataloader: training data loader
        val_dataloader: validation data loader
        config: training configuration
    """

    # Ensure model is Lightning compatible
    if not isinstance(model, LightningModule):
        print("Model is not Lightning compatible - applying wrapper...")
        model = TFTLightningWrapper(model)

    # Setup callbacks
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            min_delta=0.001,
            patience=10,
            verbose=True,
            mode="min"
        ),
        ModelCheckpoint(
            monitor="val_loss",
            dirpath="./checkpoints",
            filename="tft-{epoch:02d}-{val_loss:.4f}",
            save_top_k=1,
            mode="min"
        ),
        LearningRateMonitor(logging_interval='epoch')
    ]

    # Setup logger
    logger = TensorBoardLogger(save_dir="./lightning_logs", name="tft_training")

    # Create trainer
    trainer = Trainer(
        max_epochs=config['max_epochs'],
        callbacks=callbacks,
        logger=logger,
        gradient_clip_val=config.get('gradient_clip_val', 1.0),
        accelerator='auto',
        devices=1 if torch.cuda.is_available() else 'auto',
        precision='32-true',
        enable_checkpointing=True,
        enable_progress_bar=True,
    )

    # Train the model
    print("Starting Lightning training with wrapped model...")
    trainer.fit(
        model=model,
        train_dataloaders=train_dataloader,
        val_dataloaders=val_dataloader
    )

    return trainer, model

print("✓ Lightning compatibility wrapper classes defined")
print("Usage: wrapped_model = TFTLightningWrapper(your_tft_model)")
print("       or use create_compatible_tft_model() for automatic wrapping")

✓ Lightning compatibility wrapper classes defined
Usage: wrapped_model = TFTLightningWrapper(your_tft_model)
       or use create_compatible_tft_model() for automatic wrapping


In [ ]:
def train_tft_model(model, training_dataset, validation_dataset, config):

    """ Го тренира TFT моделот преку PyTorch Lightning. Го користе враперот, тука е логиката за early stopping,
    ако n епохи по ред нема подобрување завршува со трениране и ја евалуира најдобрата (прави checkpoints) и генерира логови.
"""

    # Required imports for the function
    import torch
    import os
    import glob
    from pytorch_lightning import Trainer, LightningModule
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
    from pytorch_lightning.loggers import TensorBoardLogger

    print("=== PRODUCTION-READY TFT TRAINING ===")

    # Clear GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")

    # Create data loaders OPTIMIZED for Colab Pro GPUs
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

    # Optimize workers and batch sizes based on GPU
    if "A100" in gpu_name or "V100" in gpu_name:
        num_workers = 4  # More workers for high-end GPUs
        prefetch_factor = 4  # More prefetching
        print("🚀 Using A100/V100 optimized dataloader settings")
    elif "T4" in gpu_name:
        num_workers = 3  # Slightly more workers for T4
        prefetch_factor = 3
        print("⚡ Using T4 optimized dataloader settings")
    else:
        num_workers = 2  # Conservative for unknown GPUs
        prefetch_factor = 2
        print("🛡️ Using conservative dataloader settings")

    train_dataloader = training_dataset.to_dataloader(
        train=True,
        batch_size=config['batch_size'],
        num_workers=num_workers,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=prefetch_factor,
    )

    # Use LARGER batch sizes for VALIDATION on better GPUs
    if "A100" in gpu_name:
        val_batch_size = min(256, config['batch_size'] * 2)  # Much larger for A100
    elif "V100" in gpu_name:
        val_batch_size = min(192, config['batch_size'] * 1.5)  # Larger for V100
    elif "T4" in gpu_name:
        val_batch_size = min(128, config['batch_size'])  # Moderate for T4
    else:
        val_batch_size = min(64, config['batch_size'])  # Conservative

    val_dataloader = validation_dataset.to_dataloader(
        train=False,
        batch_size=val_batch_size,
        num_workers=num_workers,          # ⬆️ Optimized workers
        persistent_workers=True,          # Keep workers alive for speed
        pin_memory=True,                  # Faster GPU transfer
        prefetch_factor=prefetch_factor,  # ⬆️ More prefetching for better GPUs
    )
    print(f"Training batches: {len(train_dataloader)}")
    print(f"Validation batches: {len(val_dataloader)} (batch_size={val_batch_size})")

    # FIXED: Only proceed if we have meaningful validation data
    if len(val_dataloader) < 2:
        print(f"CRITICAL ERROR: Insufficient validation batches ({len(val_dataloader)})")
        print("Training requires at least 2 validation batches for stable training.")

        # Try even smaller batch size as final attempt
        if val_batch_size > 1:
            print("Attempting with batch_size=1 for validation...")
            val_dataloader = validation_dataset.to_dataloader(
                train=False,
                batch_size=1,
                num_workers=0,
            )
            print(f"With batch_size=1: {len(val_dataloader)} validation batches")

            if len(val_dataloader) < 2:
                raise ValueError(f"Insufficient validation data for training. Only {len(val_dataloader)} batch(es) available even with batch_size=1.")
        else:
            raise ValueError(f"Insufficient validation data for training. Only {len(val_dataloader)} batch(es) available.")

    # CLEAN COMPATIBILITY FIX: Use the standalone wrapper
    print("Checking Lightning compatibility...")

    if not isinstance(model, LightningModule):
        print("🔧 Applying TFTLightningWrapper for compatibility...")
        model = TFTLightningWrapper(model)
        print(f"✓ Model wrapped. Lightning compatible: {isinstance(model, LightningModule)}")
    else:
        print("✓ Model is already Lightning compatible")

    # Production-ready callbacks
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            min_delta=config['min_delta'],
            patience=config['patience'],
            verbose=True,
            mode="min"
        ),
        ModelCheckpoint(
            monitor="val_loss",
            dirpath="./checkpoints",
            filename="best-tft-{epoch:02d}-{val_loss:.4f}",
            save_top_k=1,
            mode="min",
            save_last=True,
            verbose=True
        ),
        LearningRateMonitor(logging_interval='epoch')
    ]

    # Production-ready logger with better compatibility
    try:
        logger = TensorBoardLogger(
            save_dir="./lightning_logs",
            name="tft_experiment"
        )
    except Exception as e:
        print(f"TensorBoard logger failed, using default: {e}")
        logger = True  # Use default logger

    print("Starting PyTorch Lightning training...")
    print(f"Model type: {type(model).__name__}")
    print(f"Is LightningModule: {isinstance(model, LightningModule)}")

    # Now proceed with standard Lightning training
    if isinstance(model, LightningModule):
        print("✓ Model is compatible LightningModule - proceeding with standard training")

        # Colab Pro GPU OPTIMIZED: PyTorch Lightning Trainer for MAXIMUM PERFORMANCE
        trainer_kwargs = {
            'max_epochs': config['max_epochs'],
            'callbacks': callbacks,
            'logger': logger,
            'gradient_clip_val': config['gradient_clip_val'],
            'accelerator': 'gpu',
            'devices': 1,
            'deterministic': False,  # For better performance
            'enable_checkpointing': True,
            'enable_progress_bar': True,
            'enable_model_summary': False,  # Disabled for speed
            'precision': config['precision'],  # Mixed precision optimization
            'benchmark': True,  # Optimize for consistent input sizes
            'sync_batchnorm': False,  # Disabled for single GPU speed
        }

        # GPU-specific optimizations
        if "A100" in gpu_name:
            trainer_kwargs.update({
                'log_every_n_steps': 200,        # ⬆️ Less frequent logging for speed
                'val_check_interval': 0.25,      # ⬆️ More frequent validation
                'strategy': 'auto',              # Let Lightning choose best strategy
            })
            print("🚀 A100 optimized trainer settings applied!")
        elif "V100" in gpu_name:
            trainer_kwargs.update({
                'log_every_n_steps': 150,        # ⬆️ Less frequent logging
                'val_check_interval': 0.33,      # ⬆️ More frequent validation
                'strategy': 'auto',
            })
            print("🔥 V100 optimized trainer settings applied!")
        elif "T4" in gpu_name:
            trainer_kwargs.update({
                'log_every_n_steps': 100,        # Current T4 settings
                'val_check_interval': 0.5,       # Current validation frequency
            })
            print("⚡ T4 optimized trainer settings applied!")
        else:
            trainer_kwargs.update({
                'log_every_n_steps': 50,         # More frequent logging for unknown GPU
                'val_check_interval': 1.0,       # Conservative validation frequency
            })
            print("🛡️ Conservative trainer settings for unknown GPU")

        trainer = Trainer(**trainer_kwargs)

        # 🤖 AUTO CHECKPOINT RESTORATION: Check for existing best checkpoint
        best_checkpoint_path = None
        if os.path.exists("./checkpoints"):
            import glob
            checkpoint_pattern = os.path.join("./checkpoints", "best-tft-*.ckpt")
            existing_checkpoints = glob.glob(checkpoint_pattern)

            if existing_checkpoints:
                # Get the checkpoint with lowest validation loss from filename
                def extract_val_loss(path):
                    try:
                        filename = os.path.basename(path)
                        if "val_loss=" in filename:
                            val_loss_str = filename.split("val_loss=")[1].split(".ckpt")[0]
                            return float(val_loss_str)
                    except:
                        pass
                    return float('inf')

                best_checkpoint_path = min(existing_checkpoints, key=extract_val_loss)
                print(f"🔍 Found existing best checkpoint: {best_checkpoint_path}")

                # Test if checkpoint can be loaded with fixed loader
                try:
                    checkpoint_data = load_checkpoint_safe(best_checkpoint_path)
                    epoch = checkpoint_data.get('epoch', 'unknown')
                    print(f"✅ Checkpoint loadable! Epoch: {epoch}")
                    print(f"🔄 Will resume training from this checkpoint")
                except Exception as e:
                    print(f"❌ Checkpoint loading test failed: {e}")
                    best_checkpoint_path = None

        try:
            # PRODUCTION-READY: Standard Lightning training with auto-resume
            if best_checkpoint_path:
                print(f"🤖 RESUMING training from: {best_checkpoint_path}")
                trainer.fit(
                    model=model,
                    train_dataloaders=train_dataloader,
                    val_dataloaders=val_dataloader,
                    ckpt_path=best_checkpoint_path  # 🤖 AUTO RESUME
                )
                print("✅ Successfully resumed from checkpoint!")
            else:
                print("🆕 Starting fresh training (no checkpoint to resume)")
                trainer.fit(
                    model=model,
                    train_dataloaders=train_dataloader,
                    val_dataloaders=val_dataloader
                )

            print("✅ Production-ready training completed successfully!")

            # Extract the original model if it was wrapped
            final_model = model.model if hasattr(model, 'model') else model

            # FIXED checkpoint loading with weights_only=False for compatibility
            if hasattr(trainer, 'checkpoint_callback') and trainer.checkpoint_callback and trainer.checkpoint_callback.best_model_path:
                print(f"Loading best model from: {trainer.checkpoint_callback.best_model_path}")
                try:
                    # For TFTLightningWrapper, we need special handling
                    if isinstance(model, TFTLightningWrapper):
                        print("🔧 Loading checkpoint for wrapped TFT model...")
                        # FIXED: Load with weights_only=False to handle complex objects
                        checkpoint = torch.load(trainer.checkpoint_callback.best_model_path,
                                              map_location=model.device,
                                              weights_only=False)  # CRITICAL FIX
                        model.load_state_dict(checkpoint['state_dict'])
                        print("✅ Checkpoint loaded successfully for wrapped model")
                        return trainer, model
                    else:
                        # For unwrapped models, use standard loading
                        best_model = model.__class__.load_from_checkpoint(
                            trainer.checkpoint_callback.best_model_path
                        )
                        return trainer, best_model
                except Exception as e:
                    print(f"Could not load checkpoint: {e}")
                    print("🔧 Trying alternative loading method...")

                    # BACKUP: Try alternative loading approach
                    try:
                        # Add safe globals for Lightning objects
                        import torch.serialization
                        torch.serialization.add_safe_globals([type(model)])
                        checkpoint = torch.load(trainer.checkpoint_callback.best_model_path,
                                              map_location='cpu',
                                              weights_only=False)
                        model.load_state_dict(checkpoint['state_dict'])
                        print("✅ Alternative checkpoint loading succeeded!")
                        return trainer, model
                    except Exception as e2:
                        print(f"Alternative loading also failed: {e2}")
                        print("Using current model state instead")
                        return trainer, model
            else:
                print("⚠️  No checkpoint found, using current model")
                return trainer, model

        except Exception as e:
            print(f"Lightning training failed: {e}")
            raise e

    else:
        # Fallback - this should not happen after wrapping
        print("❌ CRITICAL: Model is still not recognized as LightningModule after wrapping")
        print("This indicates a serious compatibility issue.")

        # Check if it's a pytorch-forecasting model with Lightning methods
        has_training_step = hasattr(model, 'training_step')
        has_validation_step = hasattr(model, 'validation_step')
        has_configure_optimizers = hasattr(model, 'configure_optimizers')

        if has_training_step and has_validation_step and has_configure_optimizers:
            print("Model has Lightning methods but type checking fails")
            print("This is likely due to incompatible Lightning versions")
            print("Consider:")
            print("1. Downgrading PyTorch Lightning: !pip install pytorch-lightning==1.9.0")
            print("2. Upgrading pytorch-forecasting: !pip install --upgrade pytorch-forecasting")
            print("3. Using manual training loop")

        raise TypeError(f"Could not make model compatible with Lightning. Model type: {type(model)}, isinstance check: {isinstance(model, LightningModule)}")

In [ ]:
import torch
import numpy as np

def evaluate_model_fixed(model, dataloader, remove_zeros=True):

    """
    го евалуира тренираниот модел, ги зима предвидените и стварните вредности и
     според нив ги пресметува метриките како MAE,RMSE,R2,MAPE (стандардни за регресии)
     Directional Accuracy - колку моделот ја погодува насоката на цената.


    """

    model.eval()
    predictions = []
    actuals = []
    batch_count = 0
    error_count = 0

    print(f"🔧 Evaluating TFT model on {len(dataloader)} batches...")

    with torch.no_grad():
        for i, batch in enumerate(dataloader):

            try:
                # --- Unpack batch ---
                features, target_tensor = None, None
                if isinstance(batch, (tuple, list)) and len(batch) == 2:
                    features = batch[0]  # dict of inputs
                    if isinstance(batch[1], (tuple, list)):
                        for item in batch[1]:
                            if torch.is_tensor(item):
                                target_tensor = item
                                break
                    elif torch.is_tensor(batch[1]):
                        target_tensor = batch[1]
                else:
                    continue

                if features is None or target_tensor is None:
                    error_count += 1
                    continue

                # --- Get predictions ---
                output = model(features)
                if hasattr(output, 'prediction'):
                    pred = output.prediction
                    if pred.ndim == 3 and pred.shape[-1] > 1:
                        pred = pred[:, :, pred.shape[-1] // 2]  # median quantile
                    elif pred.ndim == 3:
                        pred = pred[:, :, 0]
                elif isinstance(output, torch.Tensor):
                    pred = output.squeeze(-1) if output.ndim == 3 and output.shape[-1] == 1 else output
                elif isinstance(output, (tuple, list)):
                    pred = output[0]
                elif isinstance(output, dict):
                    pred = output.get('prediction', output.get('output', None))
                else:
                    pred = None

                if pred is None:
                    error_count += 1
                    continue

                # --- Align shapes ---
                pred = pred.detach().cpu()
                target_tensor = target_tensor.detach().cpu()

                if pred.shape != target_tensor.shape:
                    min_len = min(pred.numel(), target_tensor.numel())
                    pred = pred.view(-1)[:min_len]
                    target_tensor = target_tensor.view(-1)[:min_len]

                # --- Store valid data ---
                valid_mask = torch.isfinite(pred) & torch.isfinite(target_tensor) & ~torch.isnan(pred) & ~torch.isnan(target_tensor)
                if valid_mask.sum() > 0:
                    predictions.append(pred[valid_mask])
                    actuals.append(target_tensor[valid_mask])
                    batch_count += 1
                else:
                    error_count += 1

            except Exception:
                error_count += 1
                continue

    print(f"\n📊 Collection Summary:")
    print(f"  Successful batches: {batch_count}/{len(dataloader)}")
    print(f"  Error batches: {error_count}/{len(dataloader)}")

    if not predictions:
        print("❌ No valid predictions collected")
        return {'Error': 'No valid predictions', 'Success_Rate': 0}, np.array([]), np.array([])

    # Combine tensors
    all_predictions = torch.cat(predictions)
    all_actuals = torch.cat(actuals)

    zero_count = 0
    if remove_zeros:
        zero_mask = all_actuals <= 0
        zero_count = zero_mask.sum().item()
        valid_mask = ~zero_mask
        all_predictions = all_predictions[valid_mask]
        all_actuals = all_actuals[valid_mask]

    # --- Metrics ---
    mae = torch.mean(torch.abs(all_predictions - all_actuals)).item()
    mse = torch.mean((all_predictions - all_actuals) ** 2).item()
    rmse = torch.sqrt(torch.tensor(mse)).item()

    actual_mean = torch.mean(all_actuals)
    ss_res = torch.sum((all_actuals - all_predictions) ** 2)
    ss_tot = torch.sum((all_actuals - actual_mean) ** 2)
    r2 = (1 - (ss_res / ss_tot)).item() if ss_tot > 1e-10 else float('-inf')

    mape_mask = all_actuals > 1e-6
    if mape_mask.sum() > 0:
        mape_values = torch.abs((all_actuals[mape_mask] - all_predictions[mape_mask]) / all_actuals[mape_mask]) * 100
        reasonable_mape = mape_values[mape_values < 1000]
        mape = torch.mean(reasonable_mape).item() if len(reasonable_mape) > 0 else 999.9
    else:
        mape = 999.9

    median_ae = torch.median(torch.abs(all_predictions - all_actuals)).item()

    # --- Directional Accuracy ---
    if len(all_actuals) > 1:
        actual_diff = torch.sign(all_actuals[1:] - all_actuals[:-1])
        pred_diff = torch.sign(all_predictions[1:] - all_predictions[:-1])
        directional_acc = (actual_diff == pred_diff).float().mean().item()
    else:
        directional_acc = float('nan')

    metrics = {
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2,
        'MAPE': min(mape, 999.9),
        'MedianAE': median_ae,
        'Directional_Accuracy': directional_acc * 100,  # %
        'Valid_Points': len(all_predictions),
        'Zero_Removed': zero_count,
        'Success_Rate': batch_count / len(dataloader) * 100
    }

    print(f"\n🎯 Evaluation Results:")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  Median AE: {median_ae:.4f}")
    print(f"  R²: {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  Directional Accuracy: {directional_acc*100:.2f}%")
    print(f"  Valid Points: {len(all_predictions):,}")
    print(f"  Zero Removed: {zero_count}")
    print(f"  Success Rate: {metrics['Success_Rate']:.1f}%")

    return metrics, all_predictions.numpy(), all_actuals.numpy()


In [ ]:
def analyze_feature_importance(model, training_dataset):
    """ги издвојува 15 најважни параметри"""

    try:
        # Get feature importance from model
        interpretation = model.interpret_output(
            training_dataset.to_dataloader(train=False, batch_size=64).__iter__().__next__()[0]
        )

        # Extract variable importance
        if 'variable_importances' in interpretation:
            importance_df = pd.DataFrame({
                'feature': training_dataset.reals + training_dataset.categoricals,
                'importance': interpretation['variable_importances'].cpu().numpy()
            }).sort_values('importance', ascending=False)

            print("\n=== Top 15 Most Important Features ===")
            print(importance_df.head(15))

            # Plot feature importance
            plt.figure(figsize=(12, 8))
            top_features = importance_df.head(20)
            plt.barh(range(len(top_features)), top_features['importance'])
            plt.yticks(range(len(top_features)), top_features['feature'])
            plt.xlabel('Importance')
            plt.title('Top 20 Feature Importances')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()

            return importance_df
        else:
            print("Feature importance not available in model interpretation")
            return None

    except Exception as e:
        print(f"Error extracting feature importance: {e}")
        return None

In [ ]:
def plot_predictions(actual, predicted, title="Model Predictions"):
    """прави график со предвидените/вистинските цени"""

    plt.figure(figsize=(15, 10))

    # Time series plot
    plt.subplot(2, 2, 1)
    plt.plot(actual[:500], label='Actual', alpha=0.7)
    plt.plot(predicted[:500], label='Predicted', alpha=0.7)
    plt.title(f'{title} - Time Series (First 500 points)')
    plt.legend()
    plt.grid(True)

    # Scatter plot
    plt.subplot(2, 2, 2)
    plt.scatter(actual, predicted, alpha=0.5)
    plt.plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'r--', lw=2)
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Actual vs Predicted')
    plt.grid(True)

    # Residuals
    residuals = actual - predicted
    plt.subplot(2, 2, 3)
    plt.scatter(predicted, residuals, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted')
    plt.ylabel('Residuals')
    plt.title('Residual Plot')
    plt.grid(True)

    # Histogram of residuals
    plt.subplot(2, 2, 4)
    plt.hist(residuals, bins=50, alpha=0.7)
    plt.xlabel('Residuals')
    plt.ylabel('Frequency')
    plt.title('Distribution of Residuals')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# Main training pipeline - ENHANCED FOR MAXIMUM PERFORMANCE
def run_tft_training(data_path, metadata_path=None, output_dir="./tft_output"):

    """пајплајнот за извршување на сите функции погоре :
1. Data handling → load_and_prepare_data_fixed()
2. Feature selection & correlation filtering → select_top_features() + remove_correlated_features()
3. Dataset creation → create_tft_datasets()
4. Model creation → create_tft_model()
5. Training → train_tft_model()
6. Evaluation → evaluate_model_fixed()
7. Feature importance → analyze_feature_importance()
8. Visualization → plot_predictions()
9. Saving outputs → torch.save() + json.dump() + CSV
"""

    print("=== ENHANCED TFT Training Pipeline for Maximum Performance ===")

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Load and prepare data - BULLETPROOF FIX
    print("\n1. Loading and preparing data with BULLETPROOF target='close' fix...")
    df, metadata = load_and_prepare_data_fixed(data_path, metadata_path)
    target_col = metadata['target']

    # OPTIMIZED feature selection - keep only strongest 32 features
    print("\n2. OPTIMIZED intelligent feature selection (top 30-35 strongest features)...")
    df_selected, top_features = select_top_features(df, target_col, n_features=35)  # Reduced from 75 to 32

    # OPTIONAL: Apply correlation filter to prevent overfitting
    print("\n2.5. OPTIONAL: Applying correlation filter to prevent overfitting...")
    df_selected, top_features = remove_correlated_features(df_selected, target_col, correlation_threshold=0.95)

    # Create enhanced datasets
    print("\n3. Creating enhanced TFT datasets...")
    training_dataset, validation_dataset = create_tft_datasets(df_selected, config, target_col, metadata)

    # Create enhanced model
    print("\n4. Creating enhanced TFT model...")
    model = create_tft_model(training_dataset, config)

    # Enhanced training
    print("\n5. Enhanced model training...")
    trainer, trained_model = train_tft_model(model, training_dataset, validation_dataset, config)

    # Evaluate model - FIXED VERSION
    print("\n6. Evaluating model with FIXED evaluation...")

    # Create dataloader for the ultra-robust evaluation function
    val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=32, num_workers=0)
    metrics, predictions, actuals = evaluate_model_fixed(trained_model, val_dataloader)

    # Feature importance
    print("\n7. Analyzing feature importance...")
    importance_df = analyze_feature_importance(trained_model, training_dataset)

    # Visualizations
    print("\n8. Creating visualizations...")
    if len(predictions) > 0 and len(actuals) > 0:
        plot_predictions(actuals, predictions, title="Enhanced TFT Model Predictions")

    # Save results
    print("\n9. Saving results...")

    # Save metrics
    with open(os.path.join(output_dir, 'enhanced_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    # Save configuration used
    with open(os.path.join(output_dir, 'config_used.json'), 'w') as f:
        json.dump(config, f, indent=2)

    # Save feature importance
    if importance_df is not None:
        importance_df.to_csv(os.path.join(output_dir, 'feature_importance.csv'), index=False)

    # Save selected features list
    with open(os.path.join(output_dir, 'selected_features.json'), 'w') as f:
        json.dump(top_features, f, indent=2)

    # Save model
    try:
        torch.save(trained_model.state_dict(), os.path.join(output_dir, 'enhanced_tft_model.pt'))
        print("Model saved successfully")
    except Exception as e:
        print(f"Warning: Could not save model: {e}")

    print(f"\n=== ENHANCED Training Complete! Results saved to {output_dir} ===")
    print("\n=== MAXIMUM PERFORMANCE OPTIMIZATIONS APPLIED ===")
    print("✓ Enhanced intelligent feature selection (75 features with multiple criteria)")
    print("✓ Quantile predictions for uncertainty estimation")
    print("✓ Mixed precision training for speed and efficiency")
    print("✓ Advanced model architecture with residual connections")
    print("✓ AdamW optimizer with weight decay and LR scheduling")
    print("✓ Enhanced data loading with workers and persistent memory")
    print("✓ Automatic categorical feature detection and encoding")
    print("✓ Static feature support for time-invariant information")
    print("✓ Group-wise normalization for better scaling")
    print("✓ Improved early stopping and checkpointing")
    print("✓ TensorBoard logging for monitoring")

    # Performance summary
    if not np.isnan(metrics.get('R2', np.nan)):
        r2_score = metrics['R2']
        if r2_score > 0.5:
            print(f"\n🎉 EXCELLENT PERFORMANCE: R² = {r2_score:.4f}")
        elif r2_score > 0.2:
            print(f"\n✅ GOOD PERFORMANCE: R² = {r2_score:.4f}")
        elif r2_score > 0:
            print(f"\n⚠️  MODERATE PERFORMANCE: R² = {r2_score:.4f}")
        else:
            print(f"\n❌ POOR PERFORMANCE: R² = {r2_score:.4f} - Consider more data or feature engineering")

    return {
        'model': trained_model,
        'trainer': trainer,
        'metrics': metrics,
        'importance': importance_df,
        'predictions': predictions,
        'actuals': actuals,
        'config_used': config,
        'selected_features': top_features
    }

In [ ]:
# 🛡️ MIXED PRECISION TROUBLESHOOTING for L4 GPU

def get_safe_precision_config(gpu_name, try_mixed_precision=False):
    """
    Get safe precision configuration for L4 GPU that avoids overflow issues

    Args:
        gpu_name: GPU name from torch.cuda.get_device_name(0)
        try_mixed_precision: Whether to attempt mixed precision (can cause overflow)
    """

    print(f"🔧 === PRECISION CONFIGURATION FOR {gpu_name} ===")

    if "L4" in gpu_name:
        if try_mixed_precision:
            precision = '16-mixed'
            gradient_clip = 0.5  # Much lower to prevent overflow
            print("⚠️  Trying mixed precision with very conservative gradient clipping")
            print("💡 If you get overflow errors, set try_mixed_precision=False")
        else:
            precision = '32'
            gradient_clip = 0.8  # Still conservative but not as extreme
            print("🛡️ Using safe 32-bit precision (recommended for stability)")
            print("✅ This avoids attention overflow issues completely")

        config_updates = {
            'precision': precision,
            'gradient_clip_val': gradient_clip,
        }

        print(f"🎯 Precision: {precision}")
        print(f"🎯 Gradient Clip: {gradient_clip}")

        return config_updates
    else:
        print(f"ℹ️  Using default precision settings for {gpu_name}")
        return {}

# Example usage:
print("🧪 Testing safe precision configurations...")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)

    # Safe configuration (recommended)
    safe_config = get_safe_precision_config(gpu_name, try_mixed_precision=False)
    print(f"\n✅ Recommended safe config: {safe_config}")

    # Mixed precision attempt (if you want to try)
    print("\n" + "="*50)
    mixed_config = get_safe_precision_config(gpu_name, try_mixed_precision=True)
    print(f"⚠️  Experimental mixed config: {mixed_config}")

    print("\n💡 RECOMMENDATION:")
    print("  • Start with 32-bit precision (safe_config)")
    print("  • If training is stable, you can try mixed precision later")
    print("  • L4 GPU is fast enough that 32-bit still gives good performance")

else:
    print("⚠️ No GPU detected")

🧪 Testing safe precision configurations...
🔧 === PRECISION CONFIGURATION FOR Tesla T4 ===
ℹ️  Using default precision settings for Tesla T4

✅ Recommended safe config: {}

🔧 === PRECISION CONFIGURATION FOR Tesla T4 ===
ℹ️  Using default precision settings for Tesla T4
⚠️  Experimental mixed config: {}

💡 RECOMMENDATION:
  • Start with 32-bit precision (safe_config)
  • If training is stable, you can try mixed precision later
  • L4 GPU is fast enough that 32-bit still gives good performance


In [ ]:

DATA_PATH = "/content/sample_data/AAPL_FIXED_clean.csv"  # Update this path
METADATA_PATH = "/content/sample_data/AAPL_FIXED_metadata_clean.json"  # Update this path (optional)
OUTPUT_DIR = "/content/sample_data/outputs"

results = run_tft_training(DATA_PATH, METADATA_PATH, OUTPUT_DIR)



=== ENHANCED TFT Training Pipeline for Maximum Performance ===

1. Loading and preparing data with BULLETPROOF target='close' fix...
💯 === SIMPLIFIED DATA LOADING WITH CURATED FEATURES ONLY ===
Loaded data shape: (157325, 49)
💯 BULLETPROOF: Target FORCED to: close
🎯 === SMART FEATURE CURATION ===
Target: close
Original features: 49
Correlation threshold: 0.99
Analyzing 46 feature columns for curation...

🔍 STEP 1: Target Leakage Detection
  ❌ LEAKED: EMA has 0.999888 correlation with target!
  ❌ LEAKED: SMA has 0.999852 correlation with target!
  📊 MODERATE: ad_line has 0.887306 correlation with target
  📊 MODERATE: ad_line_ma_10 has 0.887206 correlation with target
  ❌ LEAKED: vwma_10 has 0.999920 correlation with target!
  ❌ LEAKED: vwma_20 has 0.999845 correlation with target!
  ❌ LEAKED: pivot_point has 0.998898 correlation with target!
  ❌ LEAKED: resistance_1 has 0.995562 correlation with target!
  ❌ LEAKED: support_1 has 0.994485 correlation with target!
  ❌ LEAKED: high_lag_1 h

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 3 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
INFO:

✓ Created TFT model with 774,471 parameters
Direct Lightning compatibility: False
🔧 Applying Lightning compatibility wrapper...
✓ TFT model wrapped successfully
✓ Wrapped model Lightning compatibility: True

Model Architecture Summary:
  - Hidden size: 96
  - Attention heads: 8
  - Dropout: 0.15
  - Continuous hidden size: 22
  - Output quantiles: 7
  - Static categoricals: 0
  - Static reals: 0
  - Lightning compatible: True

5. Enhanced model training...
=== PRODUCTION-READY TFT TRAINING ===
Using CUDA device: Tesla T4
⚡ Using T4 optimized dataloader settings
Training batches: 785
Validation batches: 528 (batch_size=120)
Checking Lightning compatibility...
✓ Model is already Lightning compatible
Starting PyTorch Lightning training...
Model type: TFTLightningWrapper
Is LightningModule: True
✓ Model is compatible LightningModule - proceeding with standard training
⚡ T4 optimized trainer settings applied!
🆕 Starting fresh training (no checkpoint to resume)


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 120. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 3 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Training: |          | 0/? [00:00<?, ?it/s]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 599, in _fit_impl
    self._run(model, ckpt_path=ckpt_path)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1012, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1056, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 216, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 455, in advance
    self.epoch_loop.run(self._data_fetcher)
  File "/usr/local/lib/python3.12

TypeError: object of type 'NoneType' has no len()